# LSTM Wind Power Forecasting — Integrated Big Data + Neural Network Pipeline
   **Student:** Sebastian Briones | **Student Number:** 2026004
   **Modules:** Big Data Storage & Processing + Advanced Data Analytics

  ## Pipeline
  1. **Big Data (PySpark):** Load from HDFS -> Clean -> Feature Engineering -> EDA
  2. **Neural Network (Keras):** LSTM model for wind power forecasting

  ## Research Question
  Can an LSTM model accurately forecast next-24h wind power generation in Ireland's
  electricity grid using EirGrid data processed via Apache Spark?

  **Key references:**
  - Mostafa et al. (2022): Big Data analytics + ML for smart grid management
  - Shahid et al. (2021): A novel genetic LSTM model for wind power forecast
  - Inoubli et al. (2018): Comparative study of Big Data frameworks including Spark

In [23]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window 
import warnings             
warnings.filterwarnings("ignore") 
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Tutorial 4 NN using Pyspark
df_raw = spark.read.load('hdfs://localhost:9000/user/sbriones/ca1/eirgrid_raw/eirgrid_ALL_2014_2026.csv',
                         format="csv", header="true", sep=',', inferSchema=True) #infer schema from a kaggle example https://www.kaggle.com/code/qingyi/time-series-data-analysis-with-spark 
df_raw.printSchema()

root
 |-- EffectiveTime: timestamp (nullable = true)
 |-- SYSTEM_DEMAND: double (nullable = true)
 |-- GEN_EXP: double (nullable = true)
 |-- WIND_ACTUAL: double (nullable = true)
 |-- CO2_INTENSITY: double (nullable = true)
 |-- CO2_EMISSIONS: double (nullable = true)
 |-- solar_actual: double (nullable = true)
 |-- solar_forecast: double (nullable = true)
 |-- INTER_EWIC: double (nullable = true)
 |-- INTER_MOYLE: double (nullable = true)
 |-- INTER_NET: double (nullable = true)
 |-- INTER_GRNLK: double (nullable = true)



In [3]:
df_raw.show(5)

+-------------------+-------------+-------+-----------+-------------+-------------+------------+--------------+----------+-----------+---------+-----------+
|      EffectiveTime|SYSTEM_DEMAND|GEN_EXP|WIND_ACTUAL|CO2_INTENSITY|CO2_EMISSIONS|solar_actual|solar_forecast|INTER_EWIC|INTER_MOYLE|INTER_NET|INTER_GRNLK|
+-------------------+-------------+-------+-----------+-------------+-------------+------------+--------------+----------+-----------+---------+-----------+
|2014-01-01 00:00:00|       3758.0| 3390.0|     1386.0|        383.0|       1346.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:15:00|       3724.0| 3359.0|     1348.0|        383.0|       1335.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:30:00|       3666.0| 3301.0|     1272.0|        383.0|       1314.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:45:00|       3611.0| 3246.0|     1273.0|   

In [4]:
  # Inspect columns before cleaning                                                                                                                          
  # solar_actual and solar_forecast are empty for most of the dataset (pre-2020)                                                                             
  # GEN_EXP and CO2_EMISSIONS are not required for LSTM wind forecasting                                                                                                                                                                                                                 
                                                                                                                                                             
  # Select only relevant columns for LSTM wind power forecasting                                                                                             
  # Source: EirGrid Smart Grid Dashboard data dictionary                                                                                                     
COLS_TO_KEEP = ["EffectiveTime", "SYSTEM_DEMAND", "GEN_EXP", "WIND_ACTUAL", "CO2_INTENSITY"]                                                                          
                                                                                                                                                             
df_selected = df_raw.select(COLS_TO_KEEP)                                                                                                                                                                                                                 
df_selected.show(10) 

+-------------------+-------------+-------+-----------+-------------+
|      EffectiveTime|SYSTEM_DEMAND|GEN_EXP|WIND_ACTUAL|CO2_INTENSITY|
+-------------------+-------------+-------+-----------+-------------+
|2014-01-01 00:00:00|       3758.0| 3390.0|     1386.0|        383.0|
|2014-01-01 00:15:00|       3724.0| 3359.0|     1348.0|        383.0|
|2014-01-01 00:30:00|       3666.0| 3301.0|     1272.0|        383.0|
|2014-01-01 00:45:00|       3611.0| 3246.0|     1273.0|        391.0|
|2014-01-01 01:00:00|       3543.0| 3178.0|     1236.0|        393.0|
|2014-01-01 01:15:00|       3518.0| 3153.0|     1242.0|        396.0|
|2014-01-01 01:30:00|       3423.0| 3132.0|     1205.0|        393.0|
|2014-01-01 01:45:00|       3349.0| 3058.0|     1209.0|        398.0|
|2014-01-01 02:00:00|       3268.0| 2979.0|     1197.0|        398.0|
|2014-01-01 02:15:00|       3206.0| 2960.0|     1216.0|        388.0|
+-------------------+-------------+-------+-----------+-------------+
only showing top 10 

In [11]:
df_selected.summary().show()

+-------+-----------------+-----------------+------------------+------------------+
|summary|    SYSTEM_DEMAND|          GEN_EXP|       WIND_ACTUAL|     CO2_INTENSITY|
+-------+-----------------+-----------------+------------------+------------------+
|  count|           446986|           446986|            446986|            435622|
|   mean|4322.738687565158|4144.746920485205|1293.3869696142608|345.38946150561725|
| stddev| 860.423096864893| 791.152722551858|1001.0592075954738| 131.6331283247744|
|    min|           2176.0|           1925.0|             -59.0|              16.0|
|    25%|           3667.0|           3599.0|             445.0|             238.0|
|    50%|           4381.0|           4080.0|            1065.0|             339.0|
|    75%|           4929.0|           4628.0|            1964.0|             449.0|
|    max|           7497.0|           7774.0|            4713.0|             809.0|
+-------+-----------------+-----------------+------------------+------------

In [5]:
df_selected.count()

447516

In [5]:
#df_selected.columns = ["Effective Time", "System Demand (MW)", "Generation (MW)", "Wind Generation (MW)", "CO2 Intensity (gCO2/kWh)"]

In [7]:
df_clean = df_selected \
    .withColumnRenamed("EffectiveTime", "timestamp") \
    .withColumnRenamed("SYSTEM_DEMAND", "System Demand (MW)") \
    .withColumnRenamed("GEN_EXP", "Generation (MW)") \
    .withColumnRenamed("WIND_ACTUAL", "Wind Generation (MW)") \
    .withColumnRenamed("CO2_INTENSITY", "CO2 Intensity (gCO2/kWh)") \
    .withColumn("timestamp",    F.to_timestamp("timestamp")) \
    .dropDuplicates(["timestamp"]) \
    .dropna(subset=["Wind Generation (MW)", "System Demand (MW)", "timestamp"]) \
    .filter(F.col("Wind Generation (MW)") >= 0) \ #get rid of negative values
    .orderBy("timestamp")

In [8]:
df_clean.count()

446865

In [9]:
df_clean.show(10)

+-------------------+------------------+---------------+--------------------+------------------------+
|          timestamp|System Demand (MW)|Generation (MW)|Wind Generation (MW)|CO2 Intensity (gCO2/kWh)|
+-------------------+------------------+---------------+--------------------+------------------------+
|2014-01-01 00:00:00|            3758.0|         3390.0|              1386.0|                   383.0|
|2014-01-01 00:15:00|            3724.0|         3359.0|              1348.0|                   383.0|
|2014-01-01 00:30:00|            3666.0|         3301.0|              1272.0|                   383.0|
|2014-01-01 00:45:00|            3611.0|         3246.0|              1273.0|                   391.0|
|2014-01-01 01:00:00|            3543.0|         3178.0|              1236.0|                   393.0|
|2014-01-01 01:15:00|            3518.0|         3153.0|              1242.0|                   396.0|
|2014-01-01 01:30:00|            3423.0|         3132.0|              120

In [10]:
df_clean.summary().show()

26/10/07 10:29:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+--------------------+------------------------+
|summary|System Demand (MW)|  Generation (MW)|Wind Generation (MW)|CO2 Intensity (gCO2/kWh)|
+-------+------------------+-----------------+--------------------+------------------------+
|  count|            446865|           446865|              446865|                  435504|
|   mean| 4322.776080024169|4144.859572801629|  1293.7387309366363|      345.35808855946215|
| stddev| 860.4369173338156|791.1678220777559|  1000.9664154303696|       131.6327614772791|
|    min|            2176.0|           1925.0|                 0.0|                    16.0|
|    25%|            3667.0|           3599.0|               445.0|                   237.0|
|    50%|            4381.0|           4080.0|              1065.0|                   339.0|
|    75%|            4929.0|           4628.0|              1964.0|                   449.0|
|    max|            7497.0|           7774.0|              4713.0|   

In [24]:
window_spec = Window.orderBy("timestamp")                                                                                               
                                                                                                                                          
df_features = df_clean \
    .withColumn("hour", F.hour("timestamp")) \
    .withColumn("month", F.month("timestamp")) \
    .withColumn("day_of_week", F.dayofweek("timestamp")) \
    .withColumn("is_weekend", (F.dayofweek("timestamp").isin([1, 7])).cast("int")) \
    .withColumn("wind_demand_ratio", F.col("Wind Generation (MW)") / F.col("System Demand (MW)")) \
    .withColumn("wind_lag_96", F.lag("Wind Generation (MW)", 96).over(window_spec)) \
    .withColumn("wind_lag_672",F.lag("Wind Generation (MW)", 672).over(window_spec)) \
    .dropna(subset=["wind_lag_96", "wind_lag_672"])                                                                                     
                                                                                                                                          
print(f"Rows after feature engineering: {df_features.count()}")                                                                         
df_features.printSchema()

Rows after feature engineering: 446193
root
 |-- timestamp: timestamp (nullable = true)
 |-- System Demand (MW): double (nullable = true)
 |-- Generation (MW): double (nullable = true)
 |-- Wind Generation (MW): double (nullable = true)
 |-- CO2 Intensity (gCO2/kWh): double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- wind_demand_ratio: double (nullable = true)
 |-- wind_lag_96: double (nullable = true)
 |-- wind_lag_672: double (nullable = true)



In [25]:
df_features.show(10)

+-------------------+------------------+---------------+--------------------+------------------------+----+-----+-----------+----------+-------------------+-----------+------------+
|          timestamp|System Demand (MW)|Generation (MW)|Wind Generation (MW)|CO2 Intensity (gCO2/kWh)|hour|month|day_of_week|is_weekend|  wind_demand_ratio|wind_lag_96|wind_lag_672|
+-------------------+------------------+---------------+--------------------+------------------------+----+-----+-----------+----------+-------------------+-----------+------------+
|2014-01-08 00:00:00|            3789.0|         3199.0|              1253.0|                   395.0|   0|    1|          4|         0| 0.3306941145420955|     1831.0|      1386.0|
|2014-01-08 00:15:00|            3678.0|         3161.0|              1250.0|                   398.0|   0|    1|          4|         0|0.33985861881457313|     1841.0|      1348.0|
|2014-01-08 00:30:00|            3592.0|         3088.0|              1205.0|             

In [27]:
# Wind generation by year 
print("=== Average Wind Generation by Year ===")          
df_features.groupBy("year" if "year" in [c for c in df_features.columns] else F.year("timestamp")) \
    .agg(
        F.count("Wind Generation (MW)").alias("records"),
         F.round(F.mean("Wind Generation (MW)"), 2).alias("avg_wind_MW"),
         F.round(F.max("Wind Generation (MW)"), 2).alias("max_wind_MW"),
         F.round(F.mean("wind_demand_ratio"), 4).alias("avg_wind_share")
        ).orderBy("year" if "year" in df_features.columns else F.year("timestamp")).show()

# Wind generation by month                                                                                                              
print("=== Average Wind Generation by Month ===")                                                                                       
df_features.groupBy("month").agg(
    F.round(F.mean("Wind Generation (MW)"), 2).alias("avg_wind_MW")
    ).orderBy("month").show(12)                                                                                                             
                                                                                                                                          
# Wind generation by hour                                                                                                               
print("=== Average Wind Generation by Hour ===")                                                                                        
df_features.groupBy("hour").agg(
    F.round(F.mean("Wind Generation (MW)"), 2).alias("avg_wind_MW")
    ).orderBy("hour").show(24)

=== Average Wind Generation by Year ===
+---------------+-------+-----------+-----------+--------------+
|year(timestamp)|records|avg_wind_MW|max_wind_MW|avg_wind_share|
+---------------+-------+-----------+-----------+--------------+
|           2014|  34364|     718.23|     2318.0|        0.1868|
|           2015|  35036|     939.78|     2608.0|        0.2384|
|           2016|  35117|     862.91|     2827.0|        0.2179|
|           2017|  34967|    1056.68|     3292.0|        0.2643|
|           2018|  35026|    1261.28|     3990.0|        0.3007|
|           2019|  35034|    1365.07|     3991.0|        0.3293|
|           2020|  35106|    1560.52|     4239.0|        0.3756|
|           2021|  34999|    1333.53|     4471.0|        0.3061|
|           2022|  34986|    1559.95|     4585.0|        0.3558|
|           2023|  35016|    1567.77|     4629.0|        0.3481|
|           2024|  35132|     1510.3|     4629.0|        0.3262|
|           2025|  35032|    1556.58|     4675.0| 

In [28]:
df_pandas = df_features.orderBy("timestamp").toPandas()                                                                                 
                                                                                                                                          
spark.stop()                                                                                                                            
                                                            
df_pandas.to_csv("eirgrid_processed.csv", index=False)                                                                                  
                                                            
print(f"Pipeline complete.")                                                                                                            
print(f"Shape: {df_pandas.shape}")                        
print(f"Columns: {list(df_pandas.columns)}")                                                                                            
print(df_pandas.head(3)) 

Pipeline complete.
Shape: (446193, 12)
Columns: ['timestamp', 'System Demand (MW)', 'Generation (MW)', 'Wind Generation (MW)', 'CO2 Intensity (gCO2/kWh)', 'hour', 'month', 'day_of_week', 'is_weekend', 'wind_demand_ratio', 'wind_lag_96', 'wind_lag_672']
            timestamp  System Demand (MW)  Generation (MW)  \
0 2014-01-08 00:00:00              3789.0           3199.0   
1 2014-01-08 00:15:00              3678.0           3161.0   
2 2014-01-08 00:30:00              3592.0           3088.0   

   Wind Generation (MW)  CO2 Intensity (gCO2/kWh)  hour  month  day_of_week  \
0                1253.0                     395.0     0      1            4   
1                1250.0                     398.0     0      1            4   
2                1205.0                     390.0     0      1            4   

   is_weekend  wind_demand_ratio  wind_lag_96  wind_lag_672  
0           0           0.330694       1831.0        1386.0  
1           0           0.339859       1841.0        1348